<a href="https://colab.research.google.com/github/fidlarsyn/Introduction-Machine-Learning-with-python/blob/main/BAB_2_Supervised_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Supervised Machine Learning Algorithms**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mglearn

# Dataset Sintetis (Forge & Wave)


In [ ]:
# Dataset forge untuk klasifikasi biner
X_forge, y_forge = mglearn.datasets.make_forge()

# Dataset wave untuk regresi linear
X_wave, y_wave = mglearn.datasets.make_wave(n_samples=40)

# Dataset Real-World (Breast Cancer)

In [ ]:
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()
# Dataset ini memiliki 569 sampel dan 30 fitur untuk memprediksi tumor ganas/jinak

# Dataset Boston Housing yang Diperluas

In [ ]:
# Memuat dataset Boston dengan fitur interaksi (total 104 fitur)
# Ini digunakan untuk menunjukkan fenomena overfitting pada model linear
X_boston, y_boston = mglearn.datasets.load_extended_boston()

# **k-Nearest Neighbors**
k-NN adalah algoritma berbasis instans yang memprediksi berdasarkan kemiripan dengan tetangga terdekatnya.

# k-NN Classification
Semakin kecil nilai n_neighbors, semakin kompleks modelnya (berisiko overfitting). Sebaliknya, nilai yang lebih besar menghasilkan model yang lebih sederhana.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X_forge, y_forge, random_state=0)

# Menggunakan 3 tetangga terdekat
clf = KNeighborsClassifier(n_neighbors=3)
clf.fit(X_train, y_train)

print("Prediksi set tes: {}".format(clf.predict(X_test)))
print("Akurasi set tes: {:.2f}".format(clf.score(X_test, y_test)))

# k-NN Regression
Pada regresi, prediksi adalah nilai rata-rata dari tetangga terdekat. Kita mengevaluasi performa menggunakan skor R^2 (koefisien determinasi).

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X_wave, y_wave, random_state=0)

reg = KNeighborsRegressor(n_neighbors=3)
reg.fit(X_train, y_train)

print("Skor R^2 set tes: {:.2f}".format(reg.score(X_test, y_test)))

# **Model Linear untuk Regresi dan Klasifikasi**

# Linear Regression (Ordinary Least Squares)
 Model linear bekerja baik pada data berdimensi tinggi. Namun, perhatikan perbedaan skor antara dataset sederhana (wave) dan dataset kompleks (Boston).

In [ ]:
from sklearn.linear_model import LinearRegression

# Contoh pada Boston Housing untuk menunjukkan overfitting
X_train, X_test, y_train, y_test = train_test_split(
    X_boston, y_boston, random_state=0)

lr = LinearRegression().fit(X_train, y_train)

print("Skor training (Boston): {:.2f}".format(lr.score(X_train, y_train)))
print("Skor tes (Boston): {:.2f}".format(lr.score(X_test, y_test)))
# Hasil: Skor training 0.95 vs skor tes 0.61. Ini adalah tanda nyata OVERFITTING.

# Ridge Regression (Regularisasi L2)
Ridge mencegah overfitting dengan membatasi besaran koefisien melalui parameter alpha.

In [ ]:
from sklearn.linear_model import Ridge

# Alpha=1.0 adalah default. Alpha tinggi = regularisasi lebih kuat = model lebih sederhana.
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
ridge10 = Ridge(alpha=10).fit(X_train, y_train)
ridge01 = Ridge(alpha=0.1).fit(X_train, y_train)

print("Skor tes Ridge (alpha=1.0): {:.2f}".format(ridge.score(X_test, y_test)))
print("Skor tes Ridge (alpha=0.1): {:.2f}".format(ridge01.score(X_test, y_test)))

# Lasso Regression (Regularisasi L1)
Lasso dapat menetapkan beberapa koefisien menjadi tepat nol, yang secara otomatis melakukan pemilihan fitur.

In [ ]:
from sklearn.linear_model import Lasso

# Meningkatkan max_iter diperlukan agar model mencapai konvergensi pada alpha rendah
lasso = Lasso(alpha=0.01, max_iter=100000).fit(X_train, y_train)

print("Skor training: {:.2f}".format(lasso.score(X_train, y_train)))
print("Skor tes: {:.2f}".format(lasso.score(X_test, y_test)))
print("Jumlah fitur yang digunakan: {}".format(np.sum(lasso.coef_ != 0)))

# Logistic Regression & Linear SVC
Dua model linear utama untuk klasifikasi. Parameter C mengontrol regularisasi: C tinggi = regularisasi lemah (berusaha menyesuaikan setiap titik data).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42)

# Mencoba Logistic Regression dengan C=100 (regularisasi lebih lemah)
logreg100 = LogisticRegression(C=100).fit(X_train, y_train)
print("Akurasi tes (C=100): {:.3f}".format(logreg100.score(X_test, y_test)))

# Strategi One-vs.-Rest untuk Multiclass
Digunakan untuk mengklasifikasikan lebih dari dua kategori menggunakan algoritma biner.

In [ ]:
from sklearn.datasets import make_blobs
X, y = make_blobs(random_state=42)

linear_svm = LinearSVC().fit(X, y)
print("Bentuk koefisien (n_classes, n_features):", linear_svm.coef_.shape)

# **Decision Trees**

# Membangun Pohon dan Pre-Pruning
Tanpa pembatasan, pohon akan terus tumbuh hingga mencapai akurasi 100% pada data latih (overfitting). Kita gunakan max_depth untuk membatasi pertumbuhan.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42)

# Model tanpa pemangkasan (depth tidak dibatasi)
tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
# Model dengan pemangkasan (depth dibatasi untuk generalisasi lebih baik)
tree4 = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_train, y_train)

print("Akurasi tes (tanpa pruning): {:.3f}".format(tree.score(X_test, y_test)))
print("Akurasi tes (max_depth=4): {:.3f}".format(tree4.score(X_test, y_test)))

# Analisis Pohon dan Feature Importance

In [ ]:
print("Feature importances:\n{}".format(tree4.feature_importances_))

# Tes Ekstrapolasi pada Harga RAM
Pohon keputusan memiliki keterbatasan fatal: mereka tidak bisa melakukan ekstrapolasi (memprediksi nilai di luar jangkauan data latih).

In [ ]:
from sklearn.tree import DecisionTreeRegressor
import os

# Memuat data harga RAM
ram_prices = pd.read_csv(os.path.join(mglearn.datasets.DATA_PATH, "ram_price.csv"))
data_train = ram_prices[ram_prices.date < 2000]
data_test = ram_prices[ram_prices.date >= 2000]

X_train = data_train.date[:, np.newaxis]
y_train = np.log(data_train.price) # Menggunakan log agar tren terlihat linear

tree = DecisionTreeRegressor().fit(X_train, y_train)
lr = LinearRegression().fit(X_train, y_train)

X_all = ram_prices.date[:, np.newaxis]
pred_tree = np.exp(tree.predict(X_all)) # Kembalikan ke nilai asli dengan exp()
pred_lr = np.exp(lr.predict(X_all))

# Hasil visual akan menunjukkan prediksi pohon berbentuk flat/horizontal setelah tahun 2000

# **Ensemble Method**

# Random Forests
Membangun banyak pohon dengan variasi acak (bootstrap) dan merata-ratakannya untuk mengurangi overfitting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100, random_state=0)
forest.fit(X_train, y_train) # Menggunakan data cancer
print("Akurasi tes RF: {:.3f}".format(forest.score(X_test, y_test)))

# Gradient Boosted Regression Trees (GBRT)
Membangun pohon secara serial, di mana setiap pohon baru memperbaiki kesalahan pohon sebelumnya.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# GBRT sangat sensitif terhadap learning_rate dan max_depth
gbrt = GradientBoostingClassifier(random_state=0, max_depth=1, learning_rate=0.01)
gbrt.fit(X_train, y_train)

# **Kernelized Support Vector Machines (SVM)**

# Klasifikasi dengan RBF Kernel
SVM mencari batas keputusan yang kompleks. Dua parameter kunci: gamma (lebar kernel) dan C (regularisasi).

In [ ]:
from sklearn.svm import SVC
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

svc = SVC(kernel='rbf', C=1, gamma=0.1).fit(X_train, y_train)
print("Akurasi awal SVM: {:.2f}".format(svc.score(X_test, y_test))) # Biasanya rendah sebelum scaling

# Preprocessing Data (Min-Max Scaling)
SVM sangat sensitif terhadap skala fitur. Perhatikan lonjakan akurasi setelah penskalaan manual.

In [ ]:
min_on_training = X_train.min(axis=0)
range_on_training = (X_train - min_on_training).max(axis=0)

X_train_scaled = (X_train - min_on_training) / range_on_training
X_test_scaled = (X_test - min_on_training) / range_on_training

svc = SVC().fit(X_train_scaled, y_train)
print("Akurasi SVM setelah scaling: {:.3f}".format(svc.score(X_test_scaled, y_test)))

# **Neural Networks (MLP)**

# Implementasi Dasar

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=100, noise=0.25, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

mlp = MLPClassifier(solver='l-bfgs', random_state=0, hidden_layer_sizes=[10, 10])
mlp.fit(X_train, y_train)

# Standarisasi Data
Neural Networks bekerja optimal jika fitur memiliki rata-rata 0 dan varians 1. Dalam praktiknya, kita menggunakan StandardScaler.

In [ ]:
mean_on_train = X_train.mean(axis=0)
std_on_train = X_train.std(axis=0)

X_train_scaled = (X_train - mean_on_train) / std_on_train
X_test_scaled = (X_test - mean_on_train) / std_on_train

mlp = MLPClassifier(random_state=0, max_iter=1000).fit(X_train_scaled, y_train)
print("Akurasi MLP setelah standarisasi: {:.3f}".format(mlp.score(X_test_scaled, y_test)))

# **Evaluasi dan Estimasi Ketidakpastian (Uncertainty)**
Hampir semua model di scikit-learn berbagi antarmuka yang sama. Selain akurasi, kita sering kali perlu tahu seberapa yakin model terhadap prediksinya.

In [ ]:
# Pola Dasar: Fit -> Predict -> Score
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = model.score(X_test, y_test)

# Estimasi Ketidakpastian (Hanya pada Klasifikasi)
# 1. Decision Function (Jarak ke batas keputusan)
print("Decision Function:\n", logreg100.decision_function(X_test)[:5])

# 2. Predict Proba (Probabilitas untuk setiap kelas)
print("Predicted Probabilities:\n", logreg100.predict_proba(X_test)[:5])
